# PriorModel — Colab v8 resolution probe

Upload **this** notebook (`notebooks/colab_setup.ipynb` from your machine). GitHub `master` still has `dx=160 m` until those commits are pushed; a later cell patches the clone to **dx=80 m, nx=240**.

**This run:** n=250 pairs, 25 epochs, `models/res_test_v8.jld2`. Not production (n=1000). T4 16 GB is enough (~1.6 GB estimated).

**Before anything else:** `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is enough).

Keep the kernel as **Python 3**. Julia is invoked with `!julia --project=/content/PriorModel ...`.

VFSA COMMEMI eval still runs **locally** after you download the checkpoint (`scripts/evaluate_mid_scale_v8.jl`). Colab cannot load MTGeophysics/GLMakie for inversion.

## Do not use `--project=.`

Colab's working directory is `/content`. Running `julia --project=.` from there creates a leftover **`/content/Project.toml`** that hides this repository's environment. After that, `using HDF5` and `using LuxCUDA` fail with "package not found in current path".

Always pass the clone path:

```bash
julia --project=/content/PriorModel …
```

If `/content/Project.toml` already exists, delete it in the next cell.

In [ ]:
%%bash
set -euo pipefail
if [[ -f /content/Project.toml ]]; then
  echo "Removing leftover /content/Project.toml (created by --project=.)"
  rm -f /content/Project.toml /content/Manifest.toml
fi
ls -la /content/Project.toml 2>/dev/null || echo "No /content/Project.toml — good."

## Install Julia 1.12.4

In [ ]:
%%bash
set -euo pipefail
JULIA_VERSION="1.12.4"
JULIA_VER="${JULIA_VERSION%.*}"
if julia --version 2>/dev/null | grep -q "${JULIA_VERSION}"; then
  julia --version
  exit 0
fi
echo "Installing Julia ${JULIA_VERSION}…"
URL="https://julialang-s3.julialang.org/bin/linux/x64/${JULIA_VER}/julia-${JULIA_VERSION}-linux-x86_64.tar.gz"
wget -q "${URL}" -O /tmp/julia.tar.gz
tar -xzf /tmp/julia.tar.gz -C /usr/local --strip-components=1
rm /tmp/julia.tar.gz
julia --version

## Clone the repository

In [ ]:
%%bash
set -euo pipefail
# Public repo: never wait for GitHub username/password (that hangs this cell).
export GIT_TERMINAL_PROMPT=0
export GIT_PAGER=cat
unset GIT_ASKPASS SSH_ASKPASS

REPO_URL="https://github.com/hayrunnisayildiz/PriorModel.git"
ZIP_URL="https://github.com/hayrunnisayildiz/PriorModel/archive/refs/heads/master.zip"
DEST="/content/PriorModel"

echo "=== public clone (no login) ==="

if [[ -d "${DEST}/.git" ]]; then
  echo "Updating existing clone…"
  git -C "${DEST}" -c credential.helper= --no-pager fetch --depth 1 origin master
  git -C "${DEST}" --no-pager reset --hard FETCH_HEAD
else
  rm -rf "${DEST}"
  echo "Cloning ${REPO_URL} …"
  if timeout 90 git -c credential.helper= clone --depth 1 --single-branch --branch master --progress "${REPO_URL}" "${DEST}"; then
    echo "git clone OK"
  else
    echo "git clone stalled/failed — downloading public zip instead"
    rm -rf "${DEST}"
    wget -q --show-progress -O /tmp/PriorModel.zip "${ZIP_URL}"
    unzip -qo /tmp/PriorModel.zip -d /tmp
    mv /tmp/PriorModel-master "${DEST}"
    rm -f /tmp/PriorModel.zip
  fi
fi

test -f "${DEST}/Project.toml"
ls -l "${DEST}/Project.toml"
echo "OK"

## Patch UNET_MESH to dx=80 m

GitHub `master` still has 120 × 160 m. This cell is idempotent: if the clone already has nx=240 / dx=80, it does nothing. Re-run it after every `git reset --hard`.

In [ ]:
from pathlib import Path

p = Path("/content/PriorModel/src/synthetic/MeshParams.jl")
text = p.read_text()

old = """const UNET_MESH = MeshParams(
    120,                                                    # nx
    48,                                                     # nz
    160.0,                                                  # dx  (m)  — 19.2 km core
    25.0,                                                   # dz  (m)
    30,                                                     # n_stations
    unet_log_periods(),                                     # T ∈ [1e-3, 1e3] s
)"""

new = """const UNET_MESH = MeshParams(
    240,                                                    # nx  (was 120 at dx=160 m)
    48,                                                     # nz  — unchanged
    80.0,                                                   # dx  (m)  — 19.2 km core
    25.0,                                                   # dz  (m)  — unchanged
    30,                                                     # n_stations
    unet_log_periods(),                                     # T ∈ [1e-3, 1e3] s
)"""

if "nx  (was 120 at dx=160 m)" in text or (
    "240," in text and "80.0," in text and "UNET_MESH = MeshParams" in text
    and "160.0,                                                  # dx" not in text
):
    print("UNET_MESH already dx=80 m / nx=240")
elif old in text:
    p.write_text(text.replace(old, new, 1))
    print("patched UNET_MESH: 120×160 m → 240×80 m")
else:
    raise RuntimeError(
        "MeshParams.jl UNET_MESH block did not match. "
        "Inspect /content/PriorModel/src/synthetic/MeshParams.jl"
    )

# Fail-fast: numbers must be 240 / 80 after this cell.
blob = p.read_text()
assert "240," in blob and "80.0," in blob, blob[blob.find("const UNET_MESH"):blob.find("const UNET_MESH")+400]
print("OK", p)

## Mount Google Drive

Artifacts live under:

`/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel/`

- `data/synthetic/train_pairs_v8_res_test.h5` — v8 probe set (n=250, 48×240). Copy it here if you generated it locally; otherwise the build cell creates it with `xvfb-run`.
- `data/synthetic/train_pairs_v7.h5` — previous production set (not used for this run)

Those paths are gitignored (see `.gitignore`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Instantiate the Julia project

`MTGeophysics.jl` is pulled from GitHub tag **v0.4.2** (`Project.toml` `[sources]`), not from a local path.

**GLMakie warning:** MTGeophysics lists GLMakie as a hard dependency and imports it at package load. Instantiate still downloads it. On this headless VM, `src/pkg_setup.jl` skips auto-precompile so instantiate does not open an OpenGL context. Do **not** `using MTGeophysics` here; training must use `--commemi-every 0` (see last cell).

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul
echo "=== Pkg.instantiate (several minutes, GLMakie downloads but must not precompile) ==="
julia --project=/content/PriorModel -e '
println("julia started; instantiating…"); flush(stdout); flush(stderr)
using Pkg
Pkg.instantiate()
println("active=", Base.active_project())
flush(stdout)
'

## GPU check

Colab GPU runtime (`Runtime → Change runtime type → GPU`) is not enough by itself. Training uses GPU only if Julia prints `Device: CUDA GPU`. CPU fallback looks like `Device: CPU` or `LuxCUDA unavailable`.

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul

echo "=== Colab runtime ==="
nvidia-smi -L || { echo "nvidia-smi missing: Runtime is not GPU. Runtime → Change runtime type → GPU, then restart."; exit 1; }
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

echo
echo "=== Julia CUDA ==="
julia --project=/content/PriorModel -e '
println("julia started"); flush(stdout)
try
    using LuxCUDA
catch err
    println("LuxCUDA failed: ", err)
    println("→ training will be CPU")
    exit(1)
end
println("CUDA.functional() = ", CUDA.functional())
if CUDA.functional()
    println("GPU = ", CUDA.name(CUDA.device()))
    println("OK: training will use CUDA GPU")
else
    println("CUDA.jl loaded but no usable GPU (CUDA.functional()=false)")
    try
        CUDA.versioninfo()
    catch e
        println(e)
    end
    exit(1)
end
'

## Fetch v8 data from Drive (optional)

Copies `train_pairs_v8_res_test.h5` if it already exists on Drive. If it is missing, the **next** cell builds n=250 on the dx=80 m mesh (`xvfb-run`, ~20–30 min). Do not train on v7 — that set is 48×120.

In [ ]:
from pathlib import Path
import shutil

DRIVE_SYN = Path(
    "/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel/data/synthetic"
)
DST_DIR = Path("/content/PriorModel/data/synthetic")
NAME = "train_pairs_v8_res_test.h5"

print("=== copy v8 from Drive ===", flush=True)
print("source:", DRIVE_SYN, "exists=", DRIVE_SYN.is_dir(), flush=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

src = DRIVE_SYN / NAME
dst = DST_DIR / NAME
if src.is_file():
    print(f"copying {NAME} ({src.stat().st_size / 1e6:.1f} MB) …", flush=True)
    shutil.copy2(src, dst)
    print(f"OK {dst} ({dst.stat().st_size / 1e6:.1f} MB)", flush=True)
else:
    print(f"not on Drive: {src}", flush=True)
    print("→ next cell will build n=250 with xvfb-run", flush=True)

## Build n=250 on dx=80 m (skip if Drive already had the HDF5)

MTGeophysics imports GLMakie, so this uses `xvfb-run`. About 20–30 min on Colab CPU (~5 s/model). Skip this cell if the previous copy printed `OK ... train_pairs_v8_res_test.h5`.

In [ ]:
%%bash
set -euo pipefail
export GKSwstype=nul
export JULIA_PKG_PRECOMPILE_AUTO=0

H5=/content/PriorModel/data/synthetic/train_pairs_v8_res_test.h5
if [[ -f "$H5" ]]; then
  echo "already have $H5 ($(du -h "$H5" | cut -f1)) — skip build"
  exit 0
fi

echo "=== install xvfb (GLMakie needs a fake display) ==="
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xvfb >/dev/null

mkdir -p /content/PriorModel/data/synthetic
echo "=== build n=250  dx=80 m  (~20–30 min) ==="
script -q -c 'xvfb-run -a julia --project=/content/PriorModel \
  /content/PriorModel/scripts/build_train_pairs.jl \
  --n 250 \
  --out /content/PriorModel/data/synthetic/train_pairs_v8_res_test.h5 \
  --seed 42' /dev/null

ls -lh "$H5"

## Train v8 (25 epochs)

`--commemi-every 0` and `--no-plot` are required on Colab (GLMakie).

25 epochs only — this answers “does finer dx help?”, not a production run. First epoch compiles CUDA/Zygote (10–20 min freeze is normal). Look for `Device: CUDA GPU`, then `epoch 1/25 starting`, then `first GPU train step finished`.

In [ ]:
%%bash
set -euo pipefail
export GKSwstype=nul
export JULIA_PKG_PRECOMPILE_AUTO=1

H5=/content/PriorModel/data/synthetic/train_pairs_v8_res_test.h5
test -f "$H5" || { echo "missing $H5 — run the xvfb build cell first"; exit 1; }

echo "=== train v8  25 ep  → models/res_test_v8.jld2 ==="
# Fake a TTY so Julia line-buffers into this %%bash cell.
script -q -c "julia --project=/content/PriorModel \
  /content/PriorModel/src/training/train_mt_resistivity.jl \
  --dataset ${H5} \
  --epochs 25 \
  --output /content/PriorModel/models/res_test_v8.jld2 \
  --training-log /content/PriorModel/results/training_log_v8.csv \
  --split-json /content/PriorModel/results/train_val_split_v8.json \
  --curve-png /content/PriorModel/results/training_curve_v8.png \
  --commemi-every 0 --no-plot" /dev/null

## Copy checkpoint + HDF5 back to Drive

Download these for the local COMMEMI eval:

```bash
julia --project=. scripts/evaluate_mid_scale_v8.jl
```

That script needs `models/res_test_v8.jld2` next to the repo (or copy it from Drive).

In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/content/PriorModel")
DRIVE = Path("/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel")
rels = (
    "models/res_test_v8.jld2",
    "data/synthetic/train_pairs_v8_res_test.h5",
    "results/training_log_v8.csv",
    "results/train_val_split_v8.json",
)

print("=== copy v8 artifacts → Drive ===", flush=True)
for rel in rels:
    src = ROOT / rel
    dst = DRIVE / rel
    if not src.is_file():
        print(f"skip (missing): {src}", flush=True)
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"OK {dst} ({dst.stat().st_size / 1e6:.2f} MB)", flush=True)